# Dataset 2 — Steel Industry Energy Consumption (UCI)
## Etapa B — Python / Pandas

Pré-requisito: Etapa A no Orange (Select Columns mantendo `Usage_kWh`, variáveis de potência reativa, fatores de potência, `WeekStatus`, `Day_of_week`, `Load_Type`; checar valores únicos e ausentes; amostra aleatória de 20%; exportar CSV). Ajuste `CAMINHO_CSV` para o arquivo exportado.

In [ ]:
import pandas as pd

CAMINHO_CSV = "amostra_steel_industry.csv"  # ajustar para o nome/caminho real do arquivo exportado do Orange

df = pd.read_csv(CAMINHO_CSV)
df.columns

### 1. Renomear Usage_kWh e simplificar fatores de potência

Os nomes originais de fator de potência no dataset UCI costumam ser `Lagging_Current_Power_Factor` e `Leading_Current_Power_Factor` — ajuste conforme os nomes reais das colunas mantidas na Etapa A.

In [ ]:
df = df.rename(columns={
    "Usage_kWh": "Consumo_kWh",
    "Lagging_Current_Power_Factor": "Fator_Potencia_Atrasado",
    "Leading_Current_Power_Factor": "Fator_Potencia_Adiantado",
})
df.head()

### 2. Inspeção inicial: head(), shape, info(), describe()

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe()

### 3. Maior consumo registrado e limiar de 75%

In [ ]:
max_consumo = df["Consumo_kWh"].max()
limiar_75 = 0.75 * max_consumo

print(f"Consumo máximo: {max_consumo}")
print(f"Limiar (75% do máximo): {limiar_75}")

### 4. DataFrame de consumo acima do limiar: quantidade e percentual

In [ ]:
df_consumo_alto = df[df["Consumo_kWh"] > limiar_75]

qtd_consumo_alto = len(df_consumo_alto)
percentual_consumo_alto = qtd_consumo_alto / len(df) * 100

print(f"Registros de consumo elevado: {qtd_consumo_alto}")
print(f"Percentual sobre o total da amostra: {percentual_consumo_alto:.2f}%")

### 5. Quantos desses registros pertencem à categoria Maximum Load

In [ ]:
qtd_maximum_load = (df_consumo_alto["Load_Type"] == "Maximum_Load").sum()
percentual_maximum_load = qtd_maximum_load / qtd_consumo_alto * 100

print(f"Registros com Load_Type == 'Maximum_Load' dentro do consumo elevado: {qtd_maximum_load}")
print(f"Percentual dentro do consumo elevado: {percentual_maximum_load:.2f}%")

### 6. Limite coerente para fator de potência baixo

Observe a distribuição do fator de potência escolhido antes de fixar o limite — o valor abaixo (`0.80`) é um ponto de partida comum para "fator de potência baixo" em indústrias, mas deve ser ajustado após olhar `describe()`/histograma da coluna real.

In [ ]:
df["Fator_Potencia_Atrasado"].describe()

In [ ]:
limite_fator_potencia_baixo = 0.80  # ajustar após observar a distribuição real na célula acima
limite_fator_potencia_baixo

### 7. DataFrame com consumo elevado E fator de potência abaixo do limite

In [ ]:
df_consumo_fp_baixo = df[
    (df["Consumo_kWh"] > limiar_75) &
    (df["Fator_Potencia_Atrasado"] < limite_fator_potencia_baixo)
]

qtd_consumo_fp_baixo = len(df_consumo_fp_baixo)
percentual_consumo_fp_baixo = qtd_consumo_fp_baixo / len(df) * 100

print(f"Registros com consumo elevado E fator de potência baixo: {qtd_consumo_fp_baixo}")
print(f"Percentual sobre o total da amostra: {percentual_consumo_fp_baixo:.2f}%")

**Interpretação (preencher com os valores impressos acima antes de entregar):**

Esse segundo conjunto (consumo elevado + fator de potência baixo) merece mais atenção da equipe de energia porque combina dois problemas ao mesmo tempo: alta demanda de potência ativa **e** ineficiência no uso dessa energia. Um fator de potência baixo indica que parte da energia consumida da rede não está sendo convertida em trabalho útil (é potência reativa "desperdiçada" no sistema), o que geralmente implica:

- Maior custo, já que concessionárias costumam penalizar fator de potência abaixo de um limite contratual.
- Maior estresse na rede elétrica da planta durante justamente os picos de consumo, aumentando o risco de sobrecarga.
- Uma oportunidade concreta de intervenção (bancos de capacitores, correção de fator de potência) que tem retorno financeiro direto, diferente de apenas reduzir consumo bruto.

Consumo alto isolado pode ser simplesmente uma operação normal em carga máxima; consumo alto combinado com fator de potência baixo é o sinal de que há ineficiência a corrigir, não apenas demanda a gerenciar.